# Tech Challenge - Fase 3: Modelagem de Classificação de Atrasos

Este notebook dá continuidade à EDA e cobre os próximos passos de **feature engineering**, **seleção de variáveis** e **modelagem supervisionada de classificação**.

Objetivo do modelo:

- Prever se um voo concluído terá **atraso de chegada maior ou igual a 15 minutos**.

Premissa importante:

- Para evitar vazamento de dados, usamos principalmente variáveis conhecidas antes da operação do voo, como data, horário programado, companhia, origem, destino, distância e tempo programado.

## 1. Preparação do ambiente

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Carregamento dos dados

**Comentário:** carregamos as mesmas bases da EDA, mantendo tipos otimizados para reduzir uso de memória.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

CANDIDATE_DIRS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
    Path.home() / "Downloads",
]

def find_file(filename: str) -> Path:
    for directory in CANDIDATE_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    searched = "\n".join(str(path / filename) for path in CANDIDATE_DIRS)
    raise FileNotFoundError(f"Arquivo {filename} não encontrado. Caminhos verificados:\n{searched}")

flights_path = find_file("flights.csv")
airlines_path = find_file("airlines.csv")
airports_path = find_file("airports.csv")

flights_path, airlines_path, airports_path

In [ ]:
dtype_flights = {
    "YEAR": "int16",
    "MONTH": "int8",
    "DAY": "int8",
    "DAY_OF_WEEK": "int8",
    "AIRLINE": "category",
    "FLIGHT_NUMBER": "int32",
    "TAIL_NUMBER": "category",
    "ORIGIN_AIRPORT": "category",
    "DESTINATION_AIRPORT": "category",
    "SCHEDULED_DEPARTURE": "float32",
    "DEPARTURE_TIME": "float32",
    "DEPARTURE_DELAY": "float32",
    "TAXI_OUT": "float32",
    "WHEELS_OFF": "float32",
    "SCHEDULED_TIME": "float32",
    "ELAPSED_TIME": "float32",
    "AIR_TIME": "float32",
    "DISTANCE": "float32",
    "WHEELS_ON": "float32",
    "TAXI_IN": "float32",
    "SCHEDULED_ARRIVAL": "float32",
    "ARRIVAL_TIME": "float32",
    "ARRIVAL_DELAY": "float32",
    "DIVERTED": "int8",
    "CANCELLED": "int8",
    "CANCELLATION_REASON": "category",
    "AIR_SYSTEM_DELAY": "float32",
    "SECURITY_DELAY": "float32",
    "AIRLINE_DELAY": "float32",
    "LATE_AIRCRAFT_DELAY": "float32",
    "WEATHER_DELAY": "float32",
}

flights = pd.read_csv(flights_path, dtype=dtype_flights)
airlines = pd.read_csv(airlines_path)
airports = pd.read_csv(airports_path)

print(f"Voos: {flights.shape[0]:,} linhas")
print(f"Companhias: {airlines.shape[0]:,} linhas")
print(f"Aeroportos: {airports.shape[0]:,} linhas")

## 3. Definição do alvo

**Comentário:** o alvo escolhido é classificação binária. Um voo é considerado atrasado quando `ARRIVAL_DELAY >= 15`. Cancelados e desviados são removidos porque não possuem chegada comparável ao destino original.

In [ ]:
model_data = flights.query("CANCELLED == 0 and DIVERTED == 0").copy()
model_data["TARGET_DELAYED_ARRIVAL_15"] = (model_data["ARRIVAL_DELAY"] >= 15).astype("int8")
model_data["FLIGHT_DATE"] = pd.to_datetime(model_data[["YEAR", "MONTH", "DAY"]])

target_rate = model_data["TARGET_DELAYED_ARRIVAL_15"].mean() * 100
print(f"Voos concluídos: {len(model_data):,}")
print(f"Taxa do alvo atraso >= 15 min: {target_rate:.2f}%")

## 4. Feature engineering

**Comentário:** criamos variáveis de calendário, horário programado, período do dia, feriados, rota e informações geográficas. Essas variáveis estão disponíveis antes do voo e são candidatas úteis para previsão.

In [ ]:
def hhmm_to_minutes(value):
    if pd.isna(value):
        return np.nan
    value = int(value)
    hour = value // 100
    minute = value % 100
    if hour == 24:
        hour = 0
    if hour > 24 or minute > 59:
        return np.nan
    return hour * 60 + minute

def period_of_day(minutes):
    if pd.isna(minutes):
        return np.nan
    hour = int(minutes // 60)
    if 5 <= hour < 12:
        return "Manhã"
    if 12 <= hour < 18:
        return "Tarde"
    if 18 <= hour < 23:
        return "Noite"
    return "Madrugada"

us_federal_holidays_2015 = pd.to_datetime([
    "2015-01-01", "2015-01-19", "2015-02-16", "2015-05-25",
    "2015-07-03", "2015-07-04", "2015-09-07", "2015-10-12",
    "2015-11-11", "2015-11-26", "2015-12-25",
])

holiday_window_masks = [
    model_data["FLIGHT_DATE"].between(holiday - pd.Timedelta(days=3), holiday + pd.Timedelta(days=3))
    for holiday in us_federal_holidays_2015
]

model_data["SCHEDULED_DEPARTURE_MIN"] = model_data["SCHEDULED_DEPARTURE"].apply(hhmm_to_minutes).astype("float32")
model_data["SCHEDULED_ARRIVAL_MIN"] = model_data["SCHEDULED_ARRIVAL"].apply(hhmm_to_minutes).astype("float32")
model_data["SCHEDULED_DEPARTURE_HOUR"] = (model_data["SCHEDULED_DEPARTURE_MIN"] // 60).astype("float32")
model_data["SCHEDULED_ARRIVAL_HOUR"] = (model_data["SCHEDULED_ARRIVAL_MIN"] // 60).astype("float32")
model_data["DEPARTURE_PERIOD"] = model_data["SCHEDULED_DEPARTURE_MIN"].apply(period_of_day)
model_data["IS_WEEKEND"] = model_data["DAY_OF_WEEK"].isin([6, 7]).astype("int8")
model_data["IS_PEAK_DELAY_MONTH"] = model_data["MONTH"].isin([1, 2, 6, 7, 12]).astype("int8")
model_data["IS_US_FEDERAL_HOLIDAY"] = model_data["FLIGHT_DATE"].isin(us_federal_holidays_2015).astype("int8")
model_data["IS_HOLIDAY_WINDOW_3D"] = np.logical_or.reduce(holiday_window_masks).astype("int8")
model_data["ROUTE"] = model_data["ORIGIN_AIRPORT"].astype(str) + "-" + model_data["DESTINATION_AIRPORT"].astype(str)

airlines_renamed = airlines.rename(columns={"IATA_CODE": "AIRLINE", "AIRLINE": "AIRLINE_NAME"})
airports_origin = airports.rename(columns={
    "IATA_CODE": "ORIGIN_AIRPORT",
    "AIRPORT": "ORIGIN_AIRPORT_NAME",
    "CITY": "ORIGIN_CITY",
    "STATE": "ORIGIN_STATE",
    "COUNTRY": "ORIGIN_COUNTRY",
    "LATITUDE": "ORIGIN_LATITUDE",
    "LONGITUDE": "ORIGIN_LONGITUDE",
})
airports_dest = airports.rename(columns={
    "IATA_CODE": "DESTINATION_AIRPORT",
    "AIRPORT": "DESTINATION_AIRPORT_NAME",
    "CITY": "DESTINATION_CITY",
    "STATE": "DESTINATION_STATE",
    "COUNTRY": "DESTINATION_COUNTRY",
    "LATITUDE": "DESTINATION_LATITUDE",
    "LONGITUDE": "DESTINATION_LONGITUDE",
})

model_data = model_data.merge(airlines_renamed, on="AIRLINE", how="left", validate="many_to_one")
model_data = model_data.merge(airports_origin, on="ORIGIN_AIRPORT", how="left", validate="many_to_one")
model_data = model_data.merge(airports_dest, on="DESTINATION_AIRPORT", how="left", validate="many_to_one")

model_data.head()

## 5. Seleção inicial de features

**Comentário:** removemos colunas que vazariam informação do futuro, como atraso de partida, atraso de chegada, horários reais, tempos reais e causas do atraso. O primeiro modelo usa apenas contexto conhecido no planejamento do voo.

In [ ]:
target_col = "TARGET_DELAYED_ARRIVAL_15"

numeric_features = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "SCHEDULED_DEPARTURE_MIN",
    "SCHEDULED_ARRIVAL_MIN",
    "SCHEDULED_DEPARTURE_HOUR",
    "SCHEDULED_ARRIVAL_HOUR",
    "SCHEDULED_TIME",
    "DISTANCE",
    "IS_WEEKEND",
    "IS_PEAK_DELAY_MONTH",
    "IS_US_FEDERAL_HOLIDAY",
    "IS_HOLIDAY_WINDOW_3D",
    "ORIGIN_LATITUDE",
    "ORIGIN_LONGITUDE",
    "DESTINATION_LATITUDE",
    "DESTINATION_LONGITUDE",
]

categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ORIGIN_STATE",
    "DESTINATION_STATE",
    "DEPARTURE_PERIOD",
]

features = numeric_features + categorical_features

leakage_cols = [
    "DEPARTURE_TIME", "DEPARTURE_DELAY", "TAXI_OUT", "WHEELS_OFF",
    "ELAPSED_TIME", "AIR_TIME", "WHEELS_ON", "TAXI_IN", "ARRIVAL_TIME",
    "ARRIVAL_DELAY", "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY", "CANCELLATION_REASON",
]

print(f"Features numéricas: {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")
print(f"Total de features candidatas: {len(features)}")
print("Colunas removidas por risco de vazamento:")
print(leakage_cols)

## 6. Separação temporal e amostragem

**Comentário:** usamos separação temporal para simular uso real do modelo: treino em janeiro a setembro e teste em outubro a dezembro. A amostragem é opcional para acelerar experimentos.

In [ ]:
RANDOM_STATE = 42
TRAIN_SAMPLE_SIZE = 600_000
TEST_SAMPLE_SIZE = 250_000

train_data = model_data.query("MONTH <= 9").copy()
test_data = model_data.query("MONTH >= 10").copy()

def stratified_sample(df: pd.DataFrame, n: int | None, target: str, random_state: int) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df.copy()

    sampled_parts = []
    for _, part in df.groupby(target, observed=True):
        part_n = max(1, int(round(n * len(part) / len(df))))
        sampled_parts.append(part.sample(n=min(part_n, len(part)), random_state=random_state))

    return pd.concat(sampled_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)

train_model = stratified_sample(train_data, TRAIN_SAMPLE_SIZE, target_col, RANDOM_STATE)
test_model = stratified_sample(test_data, TEST_SAMPLE_SIZE, target_col, RANDOM_STATE)

X_train = train_model[features]
y_train = train_model[target_col]
X_test = test_model[features]
y_test = test_model[target_col]

split_summary = pd.DataFrame({
    "base": ["treino", "teste"],
    "linhas": [len(X_train), len(X_test)],
    "taxa_alvo_%": [y_train.mean() * 100, y_test.mean() * 100],
})

split_summary

## 7. Pré-processamento

**Comentário:** as variáveis numéricas recebem imputação e padronização. As categóricas são codificadas com one-hot encoding, agrupando categorias raras para controlar dimensionalidade.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 8. Modelos candidatos

**Comentário:** comparamos um modelo linear interpretável com um modelo de árvore. Como a classe positiva é minoritária, usamos pesos balanceados.

In [ ]:
models = {
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=30,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    for name, model in models.items()
}

pipelines

## 9. Treinamento e avaliação

**Comentário:** como o problema é desbalanceado, avaliamos além da acurácia. As métricas mais úteis aqui são ROC AUC, Average Precision, recall, precision e F1 da classe de atraso.

In [ ]:
def evaluate_classifier(name: str, pipeline: Pipeline, X_train, y_train, X_test, y_test) -> dict:
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    return {
        "modelo": name,
        "roc_auc": roc_auc_score(y_test, y_proba),
        "average_precision": average_precision_score(y_test, y_proba),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_atraso": precision_score(y_test, y_pred),
        "recall_atraso": recall_score(y_test, y_pred),
        "f1_atraso": f1_score(y_test, y_pred),
    }

results = []
for name, pipeline in pipelines.items():
    print(f"Treinando {name}...")
    results.append(evaluate_classifier(name, pipeline, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

## 10. Escolha do modelo

**Comentário:** inicialmente escolhemos o modelo com maior ROC AUC. Em um cenário real, a escolha também pode depender do custo de falso positivo e falso negativo.

In [ ]:
best_model_name = results_df.iloc[0]["modelo"]
best_pipeline = pipelines[best_model_name]

print(f"Modelo selecionado: {best_model_name}")
print(results_df.query("modelo == @best_model_name").T)

## 11. Relatório de classificação e matriz de confusão

**Comentário:** esta seção mostra os erros do modelo selecionado. O recall indica quantos atrasos reais foram capturados; a precision indica quantas previsões de atraso realmente atrasaram.

In [ ]:
y_pred_best = best_pipeline.predict(X_test)
y_proba_best = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_best, target_names=["Sem atraso", "Atraso >= 15 min"]))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Sem atraso", "Atraso"], yticklabels=["Sem atraso", "Atraso"])
plt.title(f"Matriz de confusão - {best_model_name}")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.show()

## 12. Curvas ROC e Precision-Recall

**Comentário:** a curva ROC avalia separação geral entre classes. A curva Precision-Recall é especialmente útil porque a classe de atraso é minoritária.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_proba_best, ax=axes[0], name=best_model_name)
axes[0].set_title("Curva ROC")

PrecisionRecallDisplay.from_predictions(y_test, y_proba_best, ax=axes[1], name=best_model_name)
axes[1].set_title("Curva Precision-Recall")

plt.tight_layout()
plt.show()

## 13. Ajuste de limiar de decisão

**Comentário:** o limiar padrão é 0,50, mas pode não ser o melhor para um problema desbalanceado. Aqui buscamos o limiar que maximiza F1 na base de teste.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_best >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision_atraso": precision_score(y_test, y_pred_threshold, zero_division=0),
        "recall_atraso": recall_score(y_test, y_pred_threshold, zero_division=0),
        "f1_atraso": f1_score(y_test, y_pred_threshold, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_results)
best_threshold_row = threshold_results.sort_values("f1_atraso", ascending=False).iloc[0]

sns.lineplot(data=threshold_results, x="threshold", y="precision_atraso", label="Precision")
sns.lineplot(data=threshold_results, x="threshold", y="recall_atraso", label="Recall")
sns.lineplot(data=threshold_results, x="threshold", y="f1_atraso", label="F1")
plt.axvline(best_threshold_row["threshold"], color="#b23a48", linestyle="--")
plt.title("Métricas por limiar de decisão")
plt.xlabel("Limiar")
plt.ylabel("Métrica")
plt.show()

best_threshold_row

## 14. Importância e seleção final de variáveis

**Comentário:** esta etapa interpreta quais variáveis mais influenciaram o modelo. Para Regressão Logística usamos magnitude dos coeficientes; para Random Forest usamos importância por impureza.

In [ ]:
def get_feature_names_from_preprocessor(preprocessor: ColumnTransformer) -> list[str]:
    num_names = preprocessor.named_transformers_["num"].get_feature_names_out(numeric_features).tolist()
    cat_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features).tolist()
    return num_names + cat_names

feature_names = get_feature_names_from_preprocessor(best_pipeline.named_steps["preprocessor"])
model_step = best_pipeline.named_steps["model"]

if hasattr(model_step, "coef_"):
    importance_values = np.abs(model_step.coef_[0])
    importance_label = "abs_coeficiente"
elif hasattr(model_step, "feature_importances_"):
    importance_values = model_step.feature_importances_
    importance_label = "importancia"
else:
    raise ValueError("Modelo selecionado não expõe coeficientes ou importâncias.")

feature_importance = pd.DataFrame({
    "feature_transformada": feature_names,
    importance_label: importance_values,
}).sort_values(importance_label, ascending=False)

top_features = feature_importance.head(25)

sns.barplot(data=top_features, y="feature_transformada", x=importance_label, color="#5271a3")
plt.title(f"Top 25 variáveis - {best_model_name}")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.show()

top_features

## 15. Conclusões da modelagem inicial

Preencha após executar o notebook:

- Modelo com melhor desempenho geral.
- ROC AUC, Average Precision, Precision, Recall e F1 da classe de atraso.
- Limiar recomendado se o objetivo for aumentar captura de atrasos.
- Variáveis mais importantes.
- Limitações: uso de apenas um ano, ausência de clima real, ausência de dados de congestionamento operacional em tempo real e possível mudança de comportamento ao longo do tempo.

Próximo passo natural:

- Criar uma etapa não supervisionada para agrupar aeroportos ou rotas com perfis semelhantes de atraso, volume, taxiamento e composição de causas.